# Advanced Regular Expression Assignment

This notebook loads the 700-record text dataset from the ZIP file and implements email, URL, and IPv4 extraction with validation. Regex is used for candidate extraction; Python validation is used for structure and numeric constraints.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import ipaddress
import re
from urllib.parse import urlsplit

ZIP_PATH = Path(r'C:/Users/ACER/Downloads/20_05_2026___a7bad75afbf22b04060b9183a6df9428.zip')
with ZipFile(ZIP_PATH) as zf:
    txt_name = next(name for name in zf.namelist() if name.endswith('.txt'))
    text = zf.read(txt_name).decode('utf-8')

lines = text.splitlines()
print(f'Records/lines loaded: {len(lines):,}')

## Task 1: Email validation

The first regex finds email-shaped candidates. The validator then rejects double `@`, empty labels, invalid domain labels, missing TLDs, and local-part dots at the beginning/end or repeated consecutively. Unicode letters are supported through Python's Unicode-aware character classes.

In [ ]:
EMAIL_CANDIDATE_RE = re.compile(r"(?<![\w@])[^\s<>\"'(),;:]+@[^\s<>\"'(),;:]+(?![\w@])", re.UNICODE)
LOCAL_RE = re.compile(r"[\w!#$%&'*+/=?^`{|}~-]+(?:\.[\w!#$%&'*+/=?^`{|}~-]+)*\Z", re.UNICODE)
LABEL_RE = re.compile(r"[A-Za-z0-9\u0080-\uffff](?:[A-Za-z0-9\u0080-\uffff-]{0,61}[A-Za-z0-9\u0080-\uffff])?\Z")
TLD_RE = re.compile(r"[A-Za-z\u0080-\uffff]{2,63}\Z")

def validate_email(value):
    if value.count('@') != 1:
        return None
    username, domain = value.rsplit('@', 1)
    labels = domain.split('.')
    if not (1 <= len(username) <= 64 and len(domain) <= 253):
        return None
    if not LOCAL_RE.fullmatch(username) or len(labels) < 2:
        return None
    if any(not LABEL_RE.fullmatch(label) for label in labels):
        return None
    if not TLD_RE.fullmatch(labels[-1]):
        return None
    return {'email': value, 'username': username, 'domain': domain, 'tld': labels[-1]}

email_candidates = sorted(set(m.group(0) for m in EMAIL_CANDIDATE_RE.finditer(text)))
valid_emails = [item for value in email_candidates if (item := validate_email(value))]
malformed_emails = [value for value in email_candidates if not validate_email(value)]
print('Valid:', len(valid_emails))
print('Malformed:', len(malformed_emails))
valid_emails[:5], malformed_emails[:5]

## Task 2: URL extraction

The candidate regex supports HTTP/HTTPS, ports, paths, and query strings. `urlsplit` separates protocol, domain, port, and path/query. Hostname validation rejects domains such as `bad_url..com`.

In [ ]:
URL_CANDIDATE_RE = re.compile(r"https?://[^\s<>\"']+", re.IGNORECASE)

def valid_hostname(hostname):
    if not hostname or len(hostname) > 253 or '..' in hostname:
        return False
    labels = hostname.rstrip('.').split('.')
    if len(labels) < 2 and hostname != 'localhost':
        return False
    return all(re.fullmatch(r'[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?', label) for label in labels)

def parse_url(value):
    value = value.rstrip('.,;:!?)]}')
    try:
        parts = urlsplit(value)
        hostname = parts.hostname
        port = parts.port
    except ValueError:
        return None
    if parts.scheme.lower() not in {'http', 'https'} or not valid_hostname(hostname):
        return None
    return {
        'url': value, 'protocol': parts.scheme.lower(), 'domain': hostname,
        'port': port, 'path': parts.path or '/', 'query': parts.query
    }

url_candidates = sorted(set(m.group(0) for m in URL_CANDIDATE_RE.finditer(text)))
valid_urls = [item for value in url_candidates if (item := parse_url(value))]
malformed_urls = [value for value in url_candidates if not parse_url(value)]
print('Valid:', len(valid_urls))
print('Malformed:', len(malformed_urls))
valid_urls[:5], malformed_urls[:5]

## Task 3: IPv4 detection

Regex finds four numeric octets. `ipaddress.ip_address` performs the important range validation, so `999.999.1.1` and `256.1.1.1` are rejected.

In [ ]:
IP_CANDIDATE_RE = re.compile(r"(?<![\w.])(?:\d{1,3}\.){3}\d{1,3}(?![\w.])")

def classify_ip(value):
    try:
        address = ipaddress.ip_address(value)
    except ValueError:
        return None
    return {'ip': value, 'type': 'private' if address.is_private else 'public'}

ip_candidates = sorted(set(m.group(0) for m in IP_CANDIDATE_RE.finditer(text)))
valid_ips = [item for value in ip_candidates if (item := classify_ip(value))]
invalid_ips = [value for value in ip_candidates if not classify_ip(value)]
print('Valid:', len(valid_ips))
print('Invalid:', len(invalid_ips))
print('Private:', sum(x['type'] == 'private' for x in valid_ips))
print('Public:', sum(x['type'] == 'public' for x in valid_ips))
valid_ips[:5], invalid_ips[:5]

## Required edge-case checks

The following tests demonstrate the required valid and invalid cases.

In [ ]:
assert validate_email('user+alias@sub.example.com') is not None
assert validate_email('nguyễn.văn.a@example.com') is not None
assert validate_email('invalid@@mail.com') is None
assert validate_email('admin@mail-server') is None
assert validate_email('a..b@gmail.com') is None
assert parse_url('https://example.com:8080/path?q=1') is not None
assert parse_url('http://bad_url..com') is None
assert classify_ip('192.168.1.1')['type'] == 'private'
assert classify_ip('8.8.8.8')['type'] == 'public'
assert classify_ip('999.999.1.1') is None
assert classify_ip('256.1.1.1') is None
print('All required checks passed.')